In [1]:
# NB_01_FHIR_Ingestion - Final Version with Pipeline Parameters
# ============================================================
# Medallion Architecture - Raw + Bronze Layer
# Features:
#   - Accepts parameters from Fabric Data Pipeline
#   - Falls back to defaults when run manually
#   - Metadata logging to pipeline_run_log Delta table
#   - Incremental ingestion with pagination
# ============================================================

# ═══════════════════════════════════════════════════════════════════
# CELL 1 — PIPELINE PARAMETERS
# ═══════════════════════════════════════════════════════════════════
# In Fabric: these values are injected by the pipeline at runtime
# Manually: falls back to the default values below

try:
    start_date        = getArgument("start_date",          "2026-04-01")
    days_to_ingest    = int(getArgument("days_to_ingest",  "3"))
    max_pages_per_day = int(getArgument("max_pages_per_day", "10"))
except:
    start_date        = "2026-04-01"
    days_to_ingest    = 3
    max_pages_per_day = 10

print("=" * 70)
print("📦 NB_01 — FHIR Ingestion | Raw + Bronze + Metadata Logging")
print("=" * 70)
print(f"▶️  Parameters received:")
print(f"   start_date        : {start_date}")
print(f"   days_to_ingest    : {days_to_ingest}")
print(f"   max_pages_per_day : {max_pages_per_day}")
print("=" * 70)


# ═══════════════════════════════════════════════════════════════════
# CELL 2 — IMPORTS
# ═══════════════════════════════════════════════════════════════════

from pyspark.sql.functions import (
    current_timestamp, lit, explode, monotonically_increasing_id
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    TimestampType, LongType, DoubleType
)
import requests
import json
import uuid
from datetime import datetime, timedelta

print("✅ Imports loaded\n")


# ═══════════════════════════════════════════════════════════════════
# CELL 3 — METADATA LOGGING SETUP
# ═══════════════════════════════════════════════════════════════════

PIPELINE_LOG_TABLE = "pipeline_run_log"

def ensure_log_table_exists():
    """
    Creates pipeline_run_log Delta table if it doesn't exist.
    Safe to call multiple times — idempotent.
    """
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {PIPELINE_LOG_TABLE} (
            run_id            STRING,
            pipeline_name     STRING,
            resource_type     STRING,
            start_date        STRING,
            days_ingested     INT,
            run_timestamp     TIMESTAMP,
            end_timestamp     TIMESTAMP,
            duration_seconds  DOUBLE,
            rows_ingested     LONG,
            pages_fetched     INT,
            bronze_rows       LONG,
            status            STRING,
            error_message     STRING,
            api_base_url      STRING,
            api_params        STRING,
            load_date         STRING,
            raw_path          STRING
        )
        USING DELTA
    """)
    print(f"✅ Log table '{PIPELINE_LOG_TABLE}' is ready\n")


def write_log_entry(
    run_id:          str,
    resource_type:   str,
    start_date:      str,
    days_ingested:   int,
    run_timestamp:   datetime,
    end_timestamp:   datetime,
    rows_ingested:   int,
    pages_fetched:   int,
    bronze_rows:     int,
    status:          str,
    error_message:   str,
    api_base_url:    str,
    api_params:      str,
    load_date:       str,
    raw_path:        str
):
    """
    Appends one row to pipeline_run_log after each resource completes.
    Called on both SUCCESS and FAILED outcomes.
    Every run is preserved — never overwritten.
    """
    duration = (end_timestamp - run_timestamp).total_seconds()

    log_data = [(
        run_id,
        "NB_01_FHIR_Ingestion",
        resource_type,
        start_date,
        days_ingested,
        run_timestamp,
        end_timestamp,
        round(duration, 2),
        rows_ingested,
        pages_fetched,
        bronze_rows,
        status,
        error_message,
        api_base_url,
        api_params,
        load_date,
        raw_path
    )]

    schema = StructType([
        StructField("run_id",            StringType(),    True),
        StructField("pipeline_name",     StringType(),    True),
        StructField("resource_type",     StringType(),    True),
        StructField("start_date",        StringType(),    True),
        StructField("days_ingested",     IntegerType(),   True),
        StructField("run_timestamp",     TimestampType(), True),
        StructField("end_timestamp",     TimestampType(), True),
        StructField("duration_seconds",  DoubleType(),    True),
        StructField("rows_ingested",     LongType(),      True),
        StructField("pages_fetched",     IntegerType(),   True),
        StructField("bronze_rows",       LongType(),      True),
        StructField("status",            StringType(),    True),
        StructField("error_message",     StringType(),    True),
        StructField("api_base_url",      StringType(),    True),
        StructField("api_params",        StringType(),    True),
        StructField("load_date",         StringType(),    True),
        StructField("raw_path",          StringType(),    True),
    ])

    log_df = spark.createDataFrame(log_data, schema=schema)
    log_df.write.mode("append").format("delta").saveAsTable(PIPELINE_LOG_TABLE)

    icon = "✅" if status == "SUCCESS" else "❌"
    print(f"   {icon} Log written → run_id: {run_id[:8]}... | "
          f"status: {status} | rows: {rows_ingested:,} | "
          f"duration: {duration:.1f}s")


# ═══════════════════════════════════════════════════════════════════
# CELL 4 — INGESTION FUNCTION
# ═══════════════════════════════════════════════════════════════════

def ingest_fhir_resource(
    resource_type:    str,
    start_date:       str = "2026-04-01",
    days_to_ingest:   int = 3,
    max_pages_per_day: int = 10
):
    """
    FHIR Resource Ingestion — Raw + Bronze + Metadata Logging

    Flow:
      1. Fetch paginated JSON from FHIR API  → Raw layer (Files/)
      2. Save raw JSON bundles as-is         → Lakehouse Files
      3. Explode entries → Delta table       → Bronze layer
      4. Write log entry                     → pipeline_run_log
    
    Parameters
    ----------
    resource_type     : FHIR resource e.g. "Patient", "Encounter"
    start_date        : first date to ingest e.g. "2026-04-01"
    days_to_ingest    : how many days of data to fetch
    max_pages_per_day : pagination cap per day (safety limit)
    """

    # ── Setup ──────────────────────────────────────────────────────
    run_id         = str(uuid.uuid4())
    run_timestamp  = datetime.utcnow()
    load_date      = run_timestamp.strftime("%Y-%m-%d")
    raw_base_path  = f"Files/Raw/{resource_type}/{load_date}/"
    base_url       = f"https://hapi.fhir.org/baseR4/{resource_type}"
    api_params_str = f"_count=50&_lastUpdated=ge{start_date}"

    print(f"\n{'═'*70}")
    print(f"🚀 [{resource_type}] Ingestion started")
    print(f"   run_id     : {run_id[:8]}...")
    print(f"   start_date : {start_date}  |  days: {days_to_ingest}  |  "
          f"max_pages/day: {max_pages_per_day}")
    print(f"   raw path   : {raw_base_path}")
    print(f"{'═'*70}")

    total_resources = 0
    total_pages     = 0

    try:

        # ── Step 1: Fetch & Save Raw JSON ──────────────────────────
        current_date = datetime.strptime(start_date, "%Y-%m-%d")
        end_date     = current_date + timedelta(days=days_to_ingest - 1)

        while current_date <= end_date:
            day_str       = current_date.strftime("%Y-%m-%d")
            day_page      = 0
            day_resources = 0

            print(f"\n   📅 Processing {day_str}")

            params = {"_count": 50, "_lastUpdated": f"ge{day_str}"}
            url    = base_url

            while url and day_page < max_pages_per_day:
                try:
                    response = requests.get(
                        url,
                        params=params if url == base_url else None,
                        timeout=30
                    )
                    response.raise_for_status()
                    bundle = response.json()

                    day_page        += 1
                    total_pages     += 1
                    count_in_page    = len(bundle.get("entry", []))
                    day_resources   += count_in_page
                    total_resources += count_in_page

                    # Save raw bundle JSON as-is
                    file_name = (
                        f"bundle_{day_str}_page_{day_page:03d}_"
                        f"{datetime.utcnow().strftime('%H%M%S')}.json"
                    )
                    mssparkutils.fs.put(
                        f"{raw_base_path}{file_name}",
                        json.dumps(bundle, indent=2),
                        overwrite=True
                    )
                    print(f"      ✅ Page {day_page}: "
                          f"saved {count_in_page} resource(s)")

                    # Find next page URL
                    url = next(
                        (lnk["url"] for lnk in bundle.get("link", [])
                         if lnk.get("relation") == "next"),
                        None
                    )
                    params = None   # only needed on first request

                except Exception as page_err:
                    print(f"      ⚠️  Page {day_page} error: {page_err}")
                    break

            print(f"   ✅ {day_str}: {day_resources} resource(s) "
                  f"across {day_page} page(s)")
            current_date += timedelta(days=1)

        # ── Step 2: Write Bronze Delta Table ───────────────────────
        print(f"\n   🔄 Writing Bronze Delta table...")

        raw_df = (spark.read
                  .option("multiline", "true")
                  .json(f"{raw_base_path}*.json"))

        bronze_df = (
            raw_df
            .select(explode("entry").alias("entry"))
            .select("entry.resource.*")
            .withColumn("extraction_timestamp", current_timestamp())
            .withColumn("load_date",            lit(load_date))
            .withColumn("api_base_url",         lit(base_url))
            .withColumn("api_params",           lit(api_params_str))
            .withColumn("ingestion_start_date", lit(start_date))
            .withColumn("ingestion_days",       lit(days_to_ingest))
            .withColumn("bronze_load_id",       monotonically_increasing_id())
        )

        table_name  = f"bronze_{resource_type.lower()}"
        bronze_rows = bronze_df.count()

        bronze_df.write.mode("append").format("delta").saveAsTable(table_name)

        print(f"   ✅ '{table_name}' updated — {bronze_rows:,} rows written")
        display(bronze_df.limit(5))

        # ── Step 3: Log SUCCESS ────────────────────────────────────
        write_log_entry(
            run_id        = run_id,
            resource_type = resource_type,
            start_date    = start_date,
            days_ingested = days_to_ingest,
            run_timestamp = run_timestamp,
            end_timestamp = datetime.utcnow(),
            rows_ingested = total_resources,
            pages_fetched = total_pages,
            bronze_rows   = bronze_rows,
            status        = "SUCCESS",
            error_message = None,
            api_base_url  = base_url,
            api_params    = api_params_str,
            load_date     = load_date,
            raw_path      = raw_base_path
        )

        return table_name

    except Exception as fatal_err:
        # ── Step 3 (failure): Log FAILED ──────────────────────────
        print(f"\n   ❌ Fatal error for {resource_type}: {fatal_err}")

        write_log_entry(
            run_id        = run_id,
            resource_type = resource_type,
            start_date    = start_date,
            days_ingested = days_to_ingest,
            run_timestamp = run_timestamp,
            end_timestamp = datetime.utcnow(),
            rows_ingested = total_resources,
            pages_fetched = total_pages,
            bronze_rows   = 0,
            status        = "FAILED",
            error_message = str(fatal_err),
            api_base_url  = base_url,
            api_params    = api_params_str,
            load_date     = load_date,
            raw_path      = raw_base_path
        )

        raise  # re-raise so Fabric pipeline marks activity as failed


# ═══════════════════════════════════════════════════════════════════
# CELL 5 — ENTRY POINT
# ═══════════════════════════════════════════════════════════════════

print("=" * 70)
print("🚀 FHIR Medallion Architecture — Ingestion Started")
print(f"   start_date        : {start_date}")
print(f"   days_to_ingest    : {days_to_ingest}")
print(f"   max_pages_per_day : {max_pages_per_day}")
print("=" * 70 + "\n")

# Create log table once before any ingestion
ensure_log_table_exists()

# Run in required order: Patient → Encounter → Observation → Condition
for resource in ["Patient", "Encounter", "Observation", "Condition"]:
    ingest_fhir_resource(
        resource_type     = resource,
        start_date        = start_date,         # ← from pipeline parameter
        days_to_ingest    = days_to_ingest,     # ← from pipeline parameter
        max_pages_per_day = max_pages_per_day   # ← from pipeline parameter
    )

print("\n" + "=" * 70)
print("✅ All 4 resources ingested into Raw + Bronze layers!")
print("=" * 70)


# ═══════════════════════════════════════════════════════════════════
# CELL 6 — VERIFY: Show pipeline log after run
# ═══════════════════════════════════════════════════════════════════

print("\n📋 Pipeline Run Log (latest 10 entries):")
display(
    spark.table(PIPELINE_LOG_TABLE)
         .orderBy("run_timestamp", ascending=False)
         .limit(10)
)

StatementMeta(, 91ad6df7-1f6e-40ee-b795-00eed6ea40b5, 3, Finished, Available, Finished, False)

🚀 FHIR Medallion Architecture - Incremental Ingestion Started
🚀 Starting ingestion for Patient | Start Date: 2026-04-01 | Days: 3
   📅 Processing 2026-04-01
     ✅ Page 1: Saved 50 Patient(s)
     ✅ Page 2: Saved 50 Patient(s)
     ✅ Page 3: Saved 50 Patient(s)
     ✅ Page 4: Saved 50 Patient(s)
     ✅ Page 5: Saved 50 Patient(s)
     ✅ Page 6: Saved 50 Patient(s)
     ✅ Page 7: Saved 50 Patient(s)
     ✅ Page 8: Saved 50 Patient(s)
     ✅ Page 9: Saved 50 Patient(s)
     ✅ Page 10: Saved 50 Patient(s)
   ✅ Day 2026-04-01 completed: 500 resources

   📅 Processing 2026-04-02
     ✅ Page 1: Saved 50 Patient(s)
     ✅ Page 2: Saved 50 Patient(s)
     ✅ Page 3: Saved 50 Patient(s)
     ✅ Page 4: Saved 50 Patient(s)
     ✅ Page 5: Saved 50 Patient(s)
     ✅ Page 6: Saved 50 Patient(s)
     ✅ Page 7: Saved 50 Patient(s)
     ✅ Page 8: Saved 50 Patient(s)
     ✅ Page 9: Saved 50 Patient(s)
     ✅ Page 10: Saved 50 Patient(s)
   ✅ Day 2026-04-02 completed: 500 resources

   📅 Processing 2026-0

SynapseWidget(Synapse.DataFrame, 506f70b6-56ce-4719-8e1e-76bb1f226bf3)

🚀 Starting ingestion for Encounter | Start Date: 2026-04-01 | Days: 3
   📅 Processing 2026-04-01
     ✅ Page 1: Saved 50 Encounter(s)
     ✅ Page 2: Saved 50 Encounter(s)
     ✅ Page 3: Saved 50 Encounter(s)
     ✅ Page 4: Saved 50 Encounter(s)
     ✅ Page 5: Saved 50 Encounter(s)
     ✅ Page 6: Saved 50 Encounter(s)
     ✅ Page 7: Saved 50 Encounter(s)
     ✅ Page 8: Saved 50 Encounter(s)
     ✅ Page 9: Saved 50 Encounter(s)
     ✅ Page 10: Saved 50 Encounter(s)
   ✅ Day 2026-04-01 completed: 500 resources

   📅 Processing 2026-04-02
     ✅ Page 1: Saved 50 Encounter(s)
     ✅ Page 2: Saved 50 Encounter(s)
     ✅ Page 3: Saved 50 Encounter(s)
     ✅ Page 4: Saved 50 Encounter(s)
     ✅ Page 5: Saved 50 Encounter(s)
     ✅ Page 6: Saved 50 Encounter(s)
     ✅ Page 7: Saved 50 Encounter(s)
     ✅ Page 8: Saved 50 Encounter(s)
     ✅ Page 9: Saved 50 Encounter(s)
     ✅ Page 10: Saved 50 Encounter(s)
   ✅ Day 2026-04-02 completed: 500 resources

   📅 Processing 2026-04-03
     ✅ Page 1: 

SynapseWidget(Synapse.DataFrame, cf91bf5f-b605-42a3-855d-79b8ed3b175a)

🚀 Starting ingestion for Observation | Start Date: 2026-04-01 | Days: 3
   📅 Processing 2026-04-01
     ✅ Page 1: Saved 50 Observation(s)
     ✅ Page 2: Saved 50 Observation(s)
     ✅ Page 3: Saved 50 Observation(s)
     ✅ Page 4: Saved 50 Observation(s)
     ✅ Page 5: Saved 50 Observation(s)
     ✅ Page 6: Saved 50 Observation(s)
     ✅ Page 7: Saved 50 Observation(s)
     ✅ Page 8: Saved 50 Observation(s)
     ✅ Page 9: Saved 50 Observation(s)
     ✅ Page 10: Saved 50 Observation(s)
   ✅ Day 2026-04-01 completed: 500 resources

   📅 Processing 2026-04-02
     ✅ Page 1: Saved 50 Observation(s)
     ✅ Page 2: Saved 50 Observation(s)
     ✅ Page 3: Saved 50 Observation(s)
     ✅ Page 4: Saved 50 Observation(s)
     ✅ Page 5: Saved 50 Observation(s)
     ✅ Page 6: Saved 50 Observation(s)
     ✅ Page 7: Saved 50 Observation(s)
     ✅ Page 8: Saved 50 Observation(s)
     ✅ Page 9: Saved 50 Observation(s)
     ✅ Page 10: Saved 50 Observation(s)
   ✅ Day 2026-04-02 completed: 500 resources



SynapseWidget(Synapse.DataFrame, e4a43e6f-782a-49de-90c6-0a44fdbca67c)

🚀 Starting ingestion for Condition | Start Date: 2026-04-01 | Days: 3
   📅 Processing 2026-04-01
     ✅ Page 1: Saved 50 Condition(s)
     ✅ Page 2: Saved 50 Condition(s)
     ✅ Page 3: Saved 50 Condition(s)
     ✅ Page 4: Saved 50 Condition(s)
     ✅ Page 5: Saved 50 Condition(s)
     ✅ Page 6: Saved 50 Condition(s)
     ✅ Page 7: Saved 50 Condition(s)
     ✅ Page 8: Saved 50 Condition(s)
     ✅ Page 9: Saved 50 Condition(s)
     ✅ Page 10: Saved 50 Condition(s)
   ✅ Day 2026-04-01 completed: 500 resources

   📅 Processing 2026-04-02
     ✅ Page 1: Saved 50 Condition(s)
     ✅ Page 2: Saved 50 Condition(s)
     ✅ Page 3: Saved 50 Condition(s)
     ✅ Page 4: Saved 50 Condition(s)
     ✅ Page 5: Saved 50 Condition(s)
     ✅ Page 6: Saved 50 Condition(s)
     ✅ Page 7: Saved 50 Condition(s)
     ✅ Page 8: Saved 50 Condition(s)
     ✅ Page 9: Saved 50 Condition(s)
     ✅ Page 10: Saved 50 Condition(s)
   ✅ Day 2026-04-02 completed: 500 resources

   📅 Processing 2026-04-03
     ✅ Page 1: 

SynapseWidget(Synapse.DataFrame, 0fc0f354-1681-4bbb-9c13-025375e4fda2)

✅ All resources successfully ingested into Raw + Bronze layers!
You can now proceed to Silver & Gold layer transformations.


In [3]:
# Quick verification
print("=== Bronze Tables Summary ===")

for resource in ["patient", "encounter", "observation", "condition"]:
    table_name = f"bronze_{resource}"
    df = spark.sql(f"SELECT COUNT(*) as row_count FROM {table_name}")
    count = df.collect()[0][0]
    print(f"{table_name:25} → {count:,} rows")
    
    # Show sample metadata
    spark.sql(f"""
        SELECT extraction_timestamp, load_date, api_params 
        FROM {table_name} 
        LIMIT 1
    """).show(truncate=False)

StatementMeta(, 91ad6df7-1f6e-40ee-b795-00eed6ea40b5, 5, Finished, Available, Finished, False)

=== Bronze Tables Summary ===
bronze_patient            → 1,500 rows
+--------------------------+----------+-------------------------+
|extraction_timestamp      |load_date |api_params               |
+--------------------------+----------+-------------------------+
|2026-04-06 11:08:10.396536|2026-04-06|_lastUpdated=ge2026-04-01|
+--------------------------+----------+-------------------------+

bronze_encounter          → 1,500 rows
+--------------------------+----------+-------------------------+
|extraction_timestamp      |load_date |api_params               |
+--------------------------+----------+-------------------------+
|2026-04-06 11:08:41.813733|2026-04-06|_lastUpdated=ge2026-04-01|
+--------------------------+----------+-------------------------+

bronze_observation        → 1,500 rows
+--------------------------+----------+-------------------------+
|extraction_timestamp      |load_date |api_params               |
+--------------------------+----------+--------------------